### This is data analysis for zillow current housing data for Austin

In [ ]:
# Import Necessary Libraries
import numpy as np
import pandas as pd

In [ ]:
# Load in data sets

# Set names of data set files for local access
path_1 = '../data/austin_housing_data.csv'

austin_housing1_df = pd.read_csv(path_1)

### Sanity Checks

#### Data set 1

In [ ]:
austin_housing1_df.shape

In [ ]:
austin_housing1_df.head(5)

In [ ]:
austin_housing1_df.tail(5)

In [ ]:
austin_housing1_df.info()

In [ ]:
austin_housing1_df.describe()

### Clean Dataset 
Remove Unnecessary columns

In [ ]:
# Remove uneccesary columns 

# list of columns to remove
cols_to_drop = [
    "streetAddress", "homeImage", "latestPriceSource", "latest_saledate",
    "latest_salemonth", "numOfWindowFeatures", "hasSpa", "hasView",
    "numOfPhotos", "numOfAppliances", "description", "hasCooling", 
    "numOfElementarySchools", "numOfMiddleSchools", "numOfHighSchools", "avgSchoolSize", "MedianStudentsPerTeacher",
    "hasHeating", "numOfCommunityFeatures", "numOfWaterfrontFeatures", "lotSizeSqFt","numOfSecurityFeatures", "numOfPatioAndPorchFeatures",
    "numOfParkingFeatures", "numOfPrimarySchools", "numOfAccessibilityFeatures"
]

# drop columns in place
austin_housing1_df_cleaned = austin_housing1_df.drop(columns=cols_to_drop)

# check result
austin_housing1_df_cleaned.shape

In [ ]:
austin_housing1_df_cleaned.head(5)

In [ ]:
austin_housing1_df_cleaned.tail(5)

In [ ]:
austin_housing1_df_cleaned.info()

In [ ]:
austin_housing1_df_cleaned.describe()

### Clean Dataset Pt2
Remove entries marked as "Vacant Land"

In [ ]:
# drop rows where homeType is 'Vacant Land'
austin_housing1_df_cleaned = austin_housing1_df_cleaned[
    austin_housing1_df_cleaned['homeType'] != 'Vacant Land'
]

# check result
print(austin_housing1_df_cleaned['homeType'].unique())
print(austin_housing1_df_cleaned.shape)


In [ ]:
austin_housing1_df_cleaned.describe()

#### Clean Dataset Pt3
Replace single family homes listed with 0 bedrooms or 0 bathrooms with the median value

In [ ]:
# Calculate median values for Single Family homes
median_bedrooms = austin_housing1_df_cleaned.loc[
    (austin_housing1_df_cleaned['homeType'] == 'Single Family') & 
    (austin_housing1_df_cleaned['numOfBedrooms'] > 0),
    'numOfBedrooms'
].median()

median_bathrooms = austin_housing1_df_cleaned.loc[
    (austin_housing1_df_cleaned['homeType'] == 'Single Family') & 
    (austin_housing1_df_cleaned['numOfBathrooms'] > 0),
    'numOfBathrooms'
].median()

print("Single Family Home Median bedroom: ", median_bedrooms)
print("Single Family Home Median bathroom: ", median_bathrooms)

# Replace 0 bedrooms/bathrooms with the median
austin_housing1_df_cleaned.loc[
    (austin_housing1_df_cleaned['homeType'] == 'Single Family') & 
    (austin_housing1_df_cleaned['numOfBedrooms'] == 0),
    'numOfBedrooms'
] = median_bedrooms

austin_housing1_df_cleaned.loc[
    (austin_housing1_df_cleaned['homeType'] == 'Single Family') & 
    (austin_housing1_df_cleaned['numOfBathrooms'] == 0),
    'numOfBathrooms'
] = median_bathrooms

# Double-check if any Single Family homes still have 0 bedrooms/bathrooms
print(austin_housing1_df_cleaned[
    (austin_housing1_df_cleaned['homeType'] == 'Single Family') & 
    ((austin_housing1_df_cleaned['numOfBedrooms'] == 0) | 
     (austin_housing1_df_cleaned['numOfBathrooms'] == 0))
])


In [ ]:
austin_housing1_df_cleaned.shape

In [ ]:
austin_housing1_df_cleaned.describe()

In [ ]:
# Check where zero bedroom and zero bathrooms are still occuring
zero_check = austin_housing1_df_cleaned[
    (austin_housing1_df_cleaned['numOfBedrooms'] == 0) |
    (austin_housing1_df_cleaned['numOfBathrooms'] == 0)
][['homeType','numOfBedrooms','numOfBathrooms','livingAreaSqFt']]

print(zero_check.head(20))
print(zero_check['homeType'].value_counts())


In [ ]:
# Replace the zero bedroom and zero bathroom entries with the median for that hometype
# Ensure we're working on a copy
austin_housing1_df_cleaned = austin_housing1_df_cleaned.copy()

# Calculate medians for each homeType (ignoring zeros)
bedroom_medians = (
    austin_housing1_df_cleaned.loc[austin_housing1_df_cleaned['numOfBedrooms'] > 0]
    .groupby('homeType')['numOfBedrooms'].median()
)

bathroom_medians = (
    austin_housing1_df_cleaned.loc[austin_housing1_df_cleaned['numOfBathrooms'] > 0]
    .groupby('homeType')['numOfBathrooms'].median()
)

# Replace 0 bedrooms with homeType-specific median
austin_housing1_df_cleaned.loc[
    austin_housing1_df_cleaned['numOfBedrooms'] == 0, 'numOfBedrooms'
] = austin_housing1_df_cleaned.loc[
    austin_housing1_df_cleaned['numOfBedrooms'] == 0, 'homeType'
].map(bedroom_medians)

# Replace 0 bathrooms with homeType-specific median
austin_housing1_df_cleaned.loc[
    austin_housing1_df_cleaned['numOfBathrooms'] == 0, 'numOfBathrooms'
] = austin_housing1_df_cleaned.loc[
    austin_housing1_df_cleaned['numOfBathrooms'] == 0, 'homeType'
].map(bathroom_medians)

# Double-check
print(austin_housing1_df_cleaned[
    (austin_housing1_df_cleaned['numOfBedrooms'] == 0) |
    (austin_housing1_df_cleaned['numOfBathrooms'] == 0)
])


In [ ]:
austin_housing1_df_cleaned.describe()

In [ ]:
austin_housing1_df_cleaned.head()

### Split data into two data sets: Rentals and Owning

In [ ]:
# Look at price distribution to determine threshol for renting or owning

import matplotlib.pyplot as plt

# Quick look at the overall price distribution
plt.figure(figsize=(10,6))
austin_housing1_df_cleaned['latestPrice'].hist(bins=100, edgecolor='black')
plt.xlabel("Latest Price ($)")
plt.ylabel("Number of Properties")
plt.title("Distribution of Latest Prices (Rent + Sale)")
plt.xlim(0, 500000)  # zoom in on the lower end to spot rental cluster
plt.show()

# Also check descriptive stats
print(austin_housing1_df_cleaned['latestPrice'].describe())

# See the lowest 20 prices
print(austin_housing1_df_cleaned['latestPrice'].sort_values().head(20).to_list())


In [ ]:
plt.figure(figsize=(10,6))
austin_housing1_df_cleaned[austin_housing1_df_cleaned['latestPrice'] < 20000]['latestPrice'].hist(
    bins=50, edgecolor='black'
)
plt.xlabel("Latest Price ($)")
plt.ylabel("Number of Properties")
plt.title("Distribution of Rental Prices (< $20k)")
plt.show()

# Quick stats for rentals
print(austin_housing1_df_cleaned[austin_housing1_df_cleaned['latestPrice'] < 20000]['latestPrice'].describe())


#### Add new column for sale or rent to differentiate between the prices

In [ ]:
# Add a new column 'listingType' to classify as Rent or Sale
austin_housing1_df_cleaned['listingType'] = austin_housing1_df_cleaned['latestPrice'].apply(
    lambda x: 'Rent' if x < 20000 else 'Sale'
)

# Check the distribution
print(austin_housing1_df_cleaned['listingType'].value_counts())
print(austin_housing1_df_cleaned[['latestPrice','listingType']].head(10))


### Save Cleaned data to a csv

In [ ]:
# Save to CSV
austin_housing1_df_cleaned.to_csv("../data/austin_housing_cleaned.csv", index=False)